In [ ]:
import time
from pathlib import Path

from selenium import webdriver
from selenium.common.exceptions import TimeoutException
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait


def download_pastagem(
    url: str = "https://atlasdaspastagens.ufg.br/map",
    download_dir: str = "ISAGRO/IS_Agro_day/pastagem",
    ano_inicial: int = 2020,
    ano_final: int = 2023,
    headless: bool = False,
    timeout: int = 30,
):
    """Baixa dados de pastagem no Atlas de Pastagens para o intervalo de anos informado."""

    # Cria o caminho onde os CSVs baixados serao salvos.
    download_path = Path(download_dir).expanduser().resolve()
    download_path.mkdir(parents=True, exist_ok=True)

    chrome_options = webdriver.ChromeOptions()
    prefs = {
        "download.default_directory": str(download_path),
        "download.prompt_for_download": False,
        "download.directory_upgrade": True,
        "safebrowsing.enabled": True,
    }
    chrome_options.add_experimental_option("prefs", prefs)
    if headless:
        chrome_options.add_argument("--headless=new")
    chrome_options.add_argument("--start-maximized")

    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, timeout)

    # XPaths informados
    xp_camadas = "//*[@id='main']/section/div/ul[1]/li[1]/div"
    xp_ano = "//*[@id='pn_id_40_label']"
    xp_botao_download = "/html/body/app-root/app-main/section/section/div/ul[1]/div/div[2]/div[1]/div/app-layers-sidebar/div/p-accordion/div/p-accordiontab[1]/div/div[2]/div/div/div/div[1]/div[2]/div[2]/button[2]"

    def _click_xpath(xpath: str):
        elem = wait.until(EC.element_to_be_clickable((By.XPATH, xpath)))
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", elem)
        time.sleep(0.3)
        elem.click()

    try:
        driver.get(url)

        # 1) Abrir Camadas
        _click_xpath(xp_camadas)

        # 2) Iterar sobre os indices do dropdown (1 a 10)
        for i in range(1,40):
            # 3) Abrir seletor de ano
            _click_xpath(xp_ano)
            time.sleep(0.5)

            # 4) Selecionar o item do dropdown usando o indice i
            xp_ano_alvo = f"/html/body/div/div/div/div/ul/p-dropdownitem[{i}]/li/span[1]"
            # try:
            _click_xpath(xp_ano_alvo)
            # except TimeoutException:
            #     # fallback: tentar com seletor alternativo
            #     try:
            #         xp_ano_alvo_alt = f"/html/body/div/div/div/div/ul/p-dropdownitem[{i}]"
            #         _click_xpath(xp_ano_alvo_alt)
            #     except TimeoutException:
            #         # fallback: tentar com o seletor p-highlighted-option
            #         try:
            #             _click_xpath("//*[@id='p-highlighted-option']")
            #         except TimeoutException:
            #             print(f"Falha ao selecionar item {i}")
            #             continue

            # 5) Clicar no botao de download
            _click_xpath(xp_botao_download)

            # Pequeno intervalo entre downloads
            time.sleep(2)

        print(f"Downloads concluidos para os indices 1 a 40")
        print(f"Arquivos CSV salvos em: {download_path}")
        return download_path

    finally:
        # Mantem o navegador aberto por alguns segundos para inspecao visual.
        time.sleep(3)
        driver.quit()


# Exemplo de uso:
# download_pastagem()


In [ ]:
import pandas as pd
import glob

def juntar_csvs_para_excel(
    download_path: str = "/home/daianasales/vscode/ISAGRO/IS_Agro_day/pastagem/",
    output_file: str = "pastagem_consolidado.xlsx"
):
    """Junta todos os CSVs baixados em um único arquivo Excel, onde cada aba é um ano."""
    
    csv_files = glob.glob(str(Path(download_path) / "*.csv"))
    
    if not csv_files:
        print("Nenhum arquivo CSV encontrado no diretório informado.")
        return
    
    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        for csv_file in sorted(csv_files):
            # Extrai o ano do nome do arquivo
            nome_arquivo = Path(csv_file).stem
            ano = ''.join(filter(str.isdigit, nome_arquivo))[:4] or nome_arquivo
            
            df = pd.read_csv(csv_file)
            df.to_excel(writer, sheet_name=ano, index=False)
            print(f"Aba '{ano}' criada com {len(df)} linhas.")
    
    print(f"\nArquivo Excel salvo em: {output_file}")


In [5]:
download_pastagem()

Downloads concluidos para os indices 1 a 40
Arquivos CSV salvos em: /home/daianasales/vscode/ISAGRO/IS_Agro/scripts/downloads_pastagem_csv


PosixPath('/home/daianasales/vscode/ISAGRO/IS_Agro/scripts/downloads_pastagem_csv')

In [8]:
import glob
from pathlib import Path
import pandas as pd

In [14]:
def consolidar_pastagem_por_estado(
    download_path: str = "/home/daianasales/vscode/ISAGRO/IS_Agro_day/pastagem/",
    output_file: str = "pastagem_por_estado_anos.xlsx",
):
    estados = [
        ("AC", "Acre"),
        ("AL", "Alagoas"),
        ("AP", "Amapa"),
        ("AM", "Amazonas"),
        ("BA", "Bahia"),
        ("CE", "Ceara"),
        ("DF", "Distrito Federal"),
        ("ES", "Espirito Santo"),
        ("GO", "Goias"),
        ("MA", "Maranhao"),
        ("MT", "Mato Grosso"),
        ("MS", "Mato Grosso do Sul"),
        ("MG", "Minas Gerais"),
        ("PA", "Para"),
        ("PB", "Paraiba"),
        ("PR", "Parana"),
        ("PE", "Pernambuco"),
        ("PI", "Piaui"),
        ("RJ", "Rio de Janeiro"),
        ("RN", "Rio Grande do Norte"),
        ("RS", "Rio Grande do Sul"),
        ("RO", "Rondonia"),
        ("RR", "Roraima"),
        ("SC", "Santa Catarina"),
        ("SP", "Sao Paulo"),
        ("SE", "Sergipe"),
        ("TO", "Tocantins"),
    ]
    sigla_para_estado = dict(estados)
    nome_para_sigla = {nome.upper(): sigla for sigla, nome in estados}

    def extrair_ano(nome_arquivo: str, df: pd.DataFrame | None = None):
        marcador = "year="
        if marcador in nome_arquivo:
            trecho = nome_arquivo.split(marcador, 1)[1]
            ano_nome = "".join(ch for ch in trecho if ch.isdigit())[:4]
            if len(ano_nome) == 4:
                return ano_nome

        sufixo_numerico = "".join(ch for ch in nome_arquivo if ch.isdigit())
        if len(sufixo_numerico) >= 4:
            return sufixo_numerico[-4:]

        if df is not None:
            colunas = {str(c).lower().strip(): c for c in df.columns}
            if "ano" in colunas:
                anos = pd.to_numeric(df[colunas["ano"]], errors="coerce").dropna()
                if not anos.empty:
                    return str(int(anos.iloc[0]))

        return nome_arquivo

    csv_files = sorted(glob.glob(str(Path(download_path) / "*.csv")))
    if not csv_files:
        print("Nenhum CSV encontrado.")
        return None

    acumulado_por_ano = {}

    for arquivo in csv_files:
        nome = Path(arquivo).stem

        try:
            df = pd.read_csv(arquivo, sep=None, engine="python")
        except Exception:
            df = pd.read_csv(arquivo)

        ano = extrair_ano(nome, df)
        cols = {str(c).lower().strip(): c for c in df.columns}
        candidatos_estado = ["estado", "uf", "nome_uf", "sigla_uf", "unidade_federativa"]
        col_estado = next((cols[c] for c in candidatos_estado if c in cols), None)
        if col_estado is None:
            print(f"[IGNORADO] {Path(arquivo).name}: coluna de estado nao encontrada.")
            continue

        prioridade = ["area_past_ha", "pastagem", "hectare", "ha", "area", "valor"]
        col_valor = next((cols[p] for p in prioridade if p in cols), None)
        if col_valor is None:
            numericas = []
            for coluna in df.columns:
                serie_numerica = pd.to_numeric(df[coluna], errors="coerce")
                if serie_numerica.notna().any():
                    numericas.append(coluna)
            if not numericas:
                print(f"[IGNORADO] {Path(arquivo).name}: sem coluna numerica.")
                continue
            col_valor = numericas[0]

        base = df[[col_estado, col_valor]].copy()
        base[col_estado] = base[col_estado].astype(str).str.strip()
        base[col_valor] = pd.to_numeric(base[col_valor], errors="coerce").fillna(0)

        estados_normalizados = []
        for valor in base[col_estado]:
            valor_limpo = valor.upper()
            if valor_limpo in sigla_para_estado:
                estados_normalizados.append(valor_limpo)
            else:
                estados_normalizados.append(nome_para_sigla.get(valor_limpo, valor_limpo))
        base["UF"] = estados_normalizados
        base = base[base["UF"].isin(sigla_para_estado)]

        serie = base.groupby("UF", as_index=True)[col_valor].sum()

        if ano in acumulado_por_ano:
            acumulado_por_ano[ano] = acumulado_por_ano[ano].add(serie, fill_value=0)
        else:
            acumulado_por_ano[ano] = serie

    if not acumulado_por_ano:
        print("Nao foi possivel consolidar os arquivos.")
        return None

    anos_ordenados = sorted(
        acumulado_por_ano,
        key=lambda valor: int(valor) if str(valor).isdigit() else str(valor),
    )
    consolidado = pd.DataFrame(acumulado_por_ano).reindex(
        index=[sigla for sigla, _ in estados],
        columns=anos_ordenados,
        fill_value=0,
    )
    consolidado = consolidado.fillna(0)
    consolidado.index = [sigla_para_estado[sigla] for sigla in consolidado.index]
    consolidado.index.name = "Estado"
    consolidado = consolidado.reset_index()

    linha_total = {"Estado": "Total"}
    for ano in anos_ordenados:
        linha_total[ano] = consolidado[ano].sum()

    consolidado = pd.concat([consolidado, pd.DataFrame([linha_total])], ignore_index=True)

    consolidado.to_excel(output_file, index=False)
    print(f"Arquivo gerado: {output_file}")
    return consolidado

In [15]:
df_estados_anos = consolidar_pastagem_por_estado("/home/daianasales/vscode/ISAGRO/IS_Agro_day/pastagem/", "pastagem_por_estado_anos.xlsx")

Arquivo gerado: pastagem_por_estado_anos.xlsx


In [18]:
df_estados_anos.head(40)

,Estado,1985,1986,1987,1988,1989,1990,1991,1992,1993,...,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023
0,Acre,3.307905e+05,3.699826e+05,4.204052e+05,4.607275e+05,5.103593e+05,5.481282e+05,5.865734e+05,6.194714e+05,6.540435e+05,...,1.748737e+06,1.795304e+06,1.834093e+06,1.872169e+06,1.918846e+06,1.971007e+06,2.041025e+06,2.128014e+06,2.224480e+06,2.297061e+06
1,Alagoas,9.364449e+05,1.287091e+06,1.498844e+06,1.540463e+06,1.587471e+06,1.632697e+06,1.677924e+06,1.709901e+06,1.747741e+06,...,1.815927e+06,1.814464e+06,1.789413e+06,1.792592e+06,1.790928e+06,1.770147e+06,1.758647e+06,1.705095e+06,1.656630e+06,1.629038e+06
2,Amapa,7.285277e+01,5.899676e+00,3.879391e+01,7.884006e+01,1.734932e+02,5.375775e+02,8.937123e+02,7.758157e+02,4.861817e+02,...,5.273730e+04,5.460654e+04,5.284874e+04,5.228117e+04,5.462334e+04,5.618893e+04,6.120820e+04,5.895115e+04,5.709517e+04,5.744502e+04
3,Amazonas,1.211946e+05,1.452073e+05,1.792764e+05,2.098363e+05,2.323392e+05,2.398537e+05,2.504506e+05,2.666264e+05,2.865840e+05,...,1.292498e+06,1.348187e+06,1.442633e+06,1.544075e+06,1.642252e+06,1.741477e+06,1.858940e+06,1.973340e+06,2.166289e+06,2.376297e+06
4,Bahia,1.146090e+07,1.261861e+07,1.354657e+07,1.420828e+07,1.472385e+07,1.511437e+07,1.568762e+07,1.604266e+07,1.625426e+07,...,1.870522e+07,1.877742e+07,1.882886e+07,1.880112e+07,1.870978e+07,1.870148e+07,1.856659e+07,1.847174e+07,1.841445e+07,1.843004e+07
5,Ceara,4.047588e+05,5.562039e+05,7.762064e+05,9.239263e+05,1.133430e+06,1.337096e+06,1.655942e+06,1.767048e+06,1.799899e+06,...,3.515410e+06,3.650215e+06,3.717273e+06,3.742520e+06,3.733562e+06,3.706754e+06,3.678431e+06,3.686349e+06,3.725510e+06,3.796937e+06
6,Distrito Federal,7.309033e+04,7.658583e+04,8.333984e+04,8.730317e+04,9.462000e+04,1.001922e+05,1.081698e+05,1.121061e+05,1.150110e+05,...,1.030470e+05,1.046669e+05,1.090460e+05,1.079608e+05,1.054663e+05,1.063677e+05,1.065648e+05,1.045060e+05,9.619583e+04,8.549736e+04
7,Espirito Santo,2.465617e+06,2.523891e+06,2.569432e+06,2.599958e+06,2.625676e+06,2.620069e+06,2.617030e+06,2.595260e+06,2.576345e+06,...,2.260929e+06,2.261486e+06,2.273539e+06,2.270990e+06,2.244653e+06,2.209943e+06,2.179115e+06,2.122188e+06,2.082945e+06,2.068991e+06
8,Goias,1.047809e+07,1.101508e+07,1.160930e+07,1.214036e+07,1.255335e+07,1.295989e+07,1.333381e+07,1.369487e+07,1.393853e+07,...,1.490440e+07,1.478125e+07,1.471819e+07,1.461293e+07,1.442739e+07,1.424522e+07,1.405211e+07,1.390156e+07,1.375307e+07,1.370084e+07
9,Maranhao,6.482229e+05,1.142885e+06,1.620614e+06,1.952055e+06,2.248237e+06,2.500904e+06,2.717263e+06,2.979609e+06,3.226171e+06,...,8.517829e+06,8.620150e+06,8.718048e+06,8.736292e+06,8.720157e+06,8.782216e+06,8.793663e+06,8.869474e+06,9.085340e+06,9.441151e+06
